# Data Preparation for Spatial PM Analysis

This notebook prepares air quality sensor data for spatial analysis.
Since each sensor records multiple PM measurements over time, the data
is aggregated to obtain a representative PM value per sensor location.


## Objectives

- Load raw air quality sensor data
- Extract PM measurements
- Aggregate PM values per sensor location
- Create a clean dataset for spatial mapping


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import glob

# get all csv files
csv_files = glob.glob("/content/drive/MyDrive/air_quality/*.csv")
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

df.head()


,sensor_id;sensor_type;location;lat;lon;timestamp;value_type;value
0,40;SDS011;7;-1.298;36.791;2018-04-01T00:02:07....
1,40;SDS011;7;-1.298;36.791;2018-04-01T00:02:07....
2,40;SDS011;7;-1.298;36.791;2018-04-01T00:02:07....
3,40;SDS011;7;-1.298;36.791;2018-04-01T00:02:07....
4,43;SDS011;20;-1.253;36.854;2018-04-01T00:03:56...


In [ ]:
df.to_csv("/content/drive/MyDrive/air_quality/Outputs/raw_air_quality_combined.csv", index=False)

In [ ]:
import pandas as pd
air_quality = pd.read_csv("/content/drive/MyDrive/air_quality/raw_air_quality_combined.csv",sep=';', low_memory=False)
air_quality.head()

,sensor_id,sensor_type,location,lat,lon,timestamp,value_type,value
0,40,SDS011,7,-1.298,36.791,2018-04-01T00:02:07.071983+00:00,humidity,65.80
1,40,SDS011,7,-1.298,36.791,2018-04-01T00:02:07.071983+00:00,temperature,21.10
2,40,SDS011,7,-1.298,36.791,2018-04-01T00:02:07.111462+00:00,P2,4.40
3,40,SDS011,7,-1.298,36.791,2018-04-01T00:02:07.111462+00:00,P1,7.80
4,43,SDS011,20,-1.253,36.854,2018-04-01T00:03:56.860816+00:00,humidity,76.30


In [ ]:
air_quality.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15053897 entries, 0 to 15053896
Data columns (total 8 columns):
 #   Column       Dtype  
---  ------       -----  
 0   sensor_id    int64  
 1   sensor_type  object 
 2   location     int64  
 3   lat          float64
 4   lon          float64
 5   timestamp    object 
 6   value_type   object 
 7   value        object 
dtypes: float64(2), int64(2), object(4)
memory usage: 918.8+ MB


In [ ]:
air_quality.columns

Index(['sensor_id', 'sensor_type', 'location', 'lat', 'lon', 'timestamp',
       'value_type', 'value'],
      dtype='object')

In [ ]:
air_quality.dtypes

,0
sensor_id,int64
sensor_type,object
location,int64
lat,float64
lon,float64
timestamp,object
value_type,object
value,object


In [ ]:
air_quality["value_type"].head()

,value_type
0,humidity
1,temperature
2,P2
3,P1
4,humidity


## Filtering PM Measurements

Only particulate matter (PM) measurements are retained for analysis.

## Particulate Matter Representation

The dataset provides particulate matter measurements labeled as P1 and P2.
In this analysis, P2 measurements are used as a proxy for PM2.5, which represents
fine particulate matter with significant health impacts.


In [ ]:
pm_df = air_quality[air_quality["value_type"] == "P2"]
pm_df = pm_df.dropna(subset=["sensor_id", "lat", "lon", "value"])
pm_df.head()

,sensor_id,sensor_type,location,lat,lon,timestamp,value_type,value
2,40,SDS011,7,-1.298,36.791,2018-04-01T00:02:07.111462+00:00,P2,4.40
6,40,SDS011,7,-1.298,36.791,2018-04-01T00:04:36.364175+00:00,P2,4.43
8,40,SDS011,7,-1.298,36.791,2018-04-01T00:07:18.801040+00:00,P2,4.50
12,40,SDS011,7,-1.298,36.791,2018-04-01T00:09:55.795526+00:00,P2,4.87
18,40,SDS011,7,-1.298,36.791,2018-04-01T00:12:25.801304+00:00,P2,4.10


## Aggregating PM Values per Sensor

To focus on spatial patterns, PM measurements are aggregated per sensor
using the mean PM value.


In [ ]:
pm_df["value"] = pd.to_numeric(pm_df["value"], errors="coerce")
spatial_df = (
    pm_df.groupby(["sensor_id", "lat", "lon"], as_index=False).agg(mean_pm=("value", "mean"))
)

spatial_df.head()

,sensor_id,lat,lon,mean_pm
0,3,-1.311,36.767,4.890583
1,3,-1.288,36.841,18.123454
2,3,-1.259,36.799,847.404735
3,21,-1.292,36.821,10.243000
4,23,-1.298,36.791,16.374026


## Saving Clean Spatial Dataset

In [ ]:
spatial_df.to_csv("/content/drive/MyDrive/air_quality/Outputs/clean_spatial_pm.csv", index=False)